# F1LAB / Notebook 02 — Qualifying ML Backtest 🏎️

**Question:** Does learnable ML beat the very strong FP3 order baseline?

This notebook presents the experiment. Reusable dataset processing, training, feature definitions and evaluation live in `src/f1_racelab/`.

**Leakage contract:** Train 2022–23 → select winner with 2024 → report untouched 2025. All drivers from one Grand Prix stay in the same split.

**Important:** The models predict normalized **Q1** pace, then we sort within a weekend and evaluate against the *final qualifying classification*. That is a transparent V1 proxy, not a complete Q1/Q2/Q3 simulator.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from f1_racelab.models.qualifying_backtest import prepare_dataset, run_backtest, FEATURES

PROJECT_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent) if (p / 'pyproject.toml').exists()), Path.cwd())
DATASET = PROJECT_ROOT / 'data/processed/qualifying_2022_2023_2024_2025.parquet'
assert DATASET.exists(), 'Dataset missing: run uv run python -m f1_racelab.data.qualifying_dataset --years 2022 2023 2024 2025'
raw = pd.read_parquet(DATASET)
print('Rows, columns:', raw.shape)
print('Distinct weekends:', raw[['year','round']].drop_duplicates().shape[0])
display(raw.groupby('year').agg(weekends=('round','nunique'), drivers=('driver','count')))


## 1. Audit input quality

We must inspect coverage before trusting model metrics. The ingestion process intentionally logs failed weekends; historical data providers do not always return everything consistently.


In [ ]:
ingestion_log = PROJECT_ROOT / 'data/processed/qualifying_ingestion_2022_2023_2024_2025.csv'
if ingestion_log.exists():
    report = pd.read_csv(ingestion_log)
    display(report.groupby(['year','status']).size().rename('weekends').reset_index())
    display(report.loc[report.status.eq('ERROR'), ['year','round','event','detail']].head(25))
else:
    print('No ingestion CSV; first generate the dataset via CLI.')
prepared = prepare_dataset(raw)
display(prepared.groupby('year').agg(weekends=('round','nunique'), eligible_rows=('driver','count')))


## 2. What the model sees

`X` contains current-weekend FP1/2/3 pace and related practice-only signals, plus driver/circuit identifiers. Missing numerical fields are imputed **inside each training pipeline** using training-set medians.

`y = q1_gap_pct`: relative qualifying Q1 pace **after qualifying**, strictly a target. `final_position` is used for evaluation only.


In [ ]:
print('Numerical pre-qualifying features:', len(FEATURES))
print(', '.join(FEATURES))
assert 'q1_gap_pct' not in FEATURES and 'final_position' not in FEATURES
selected = ['year','round','event','driver','fp3_gap_pct','fp3_rank','q1_gap_pct','final_position']
display(prepared.loc[prepared.year.eq(2025), selected].head(20))


## 3. Train Ridge and LightGBM; keep 2025 held out

- **FP3:** sort participants by best FP3 time (no learning)
- **Ridge:** regularized linear regression → predicts Q1 gap
- **LightGBM:** gradient-boosted trees → predicts Q1 gap

We select the winner using **2024 validation MAE only**. We still show the untouched 2025 results for every model.


In [ ]:
result = run_backtest(raw)
print('Train weekends:', result['n_training_events'])
print('Validation weekends:', result['n_validation_events'])
print('Test weekends:', result['n_test_events'])
print('2024 selected method:', result['winner'])
print('2024 VALIDATION')
display(result['validation_summary'].sort_values('position_mae'))
print('2025 TEST (held out during winner selection)')
display(result['test_summary'].sort_values('position_mae'))


## 4. Which model wins race by race?

MAE is averaged per weekend first, so each Grand Prix receives equal weight regardless of missing participants. Lower MAE means fewer positions of error; higher Spearman means better ordering.


In [ ]:
events = result['test_events']
wide = events.pivot(index=['year','round','event'],columns='model', values='position_mae')
display(wide.round(2))
wide.plot(kind='line', marker='o', figsize=(12,5), title='Held-out 2025: ranking error by Grand Prix')
plt.ylabel('Mean absolute position error (lower is better)')
plt.xlabel('Event')
plt.xticks(rotation=45)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 5. Rank comparison on one held-out weekend

The scores are learned Q1-pace estimates; sorting them gives the predicted grid *within eligible drivers*. We do not use starting-grid penalties at this stage.


In [ ]:
pred = result['predictions']
test = pred[pred.split.eq('test')]
last_event = test[['year','round']].drop_duplicates().sort_values(['year','round']).iloc[-1]
event_rows = test[(test.year.eq(last_event.year)) & (test['round'].eq(last_event['round']))]
display(event_rows.pivot(index=['driver','final_position'], columns='model', values='predicted_rank').sort_index().head(25))


## 6. Worst errors and next feature ablations

We should diagnose mismatches instead of adding random features. Candidate next experiments:

1. **Historical qualifying form:** rolling driver/team gap (`shift(1)` to exclude current/future races)
2. **Teammate delta:** how much the driver tends to outperform teammate
3. **Practice → qualifying conversion:** known from *past* weekends only
4. **Sprint-specific model:** no FP3 required
5. **Phase-aware Q1/Q2/Q3 simulation:** qualify on a latent pace distribution, then Monte Carlo

Also investigate whether 3%-within-fastest filtering retains cooldowns or excludes genuinely useful representative laps.


In [ ]:
diagnostics = test.assign(position_error=lambda x: (x.predicted_rank-x.actual_rank).abs())
display(diagnostics.sort_values('position_error', ascending=False)[['model','year','round','event','driver','predicted_rank','actual_rank','position_error']].head(25))


## 7. Engineer's conclusion (fill after running)

- Which model wins 2024 validation?
- Does it still win 2025?
- How many weekends were skipped or failed ingestion?
- Which circuits show the worst mistakes?
- What signal do we add next, and how will we prove that it helps?

Do **not** describe any model as "accurate" without including the time split, baseline, metric, and coverage.
